# A Satisficing Voter Model

Python port of `sim.js`. Voters and candidates are placed uniformly in $[-1,1]^d$. True utility is $u_i(c) = -\|v_i - p_c\|_\nu^2$.

**Two imperfection parameters:**
- **Epistemic noise $t \in [0,1]$:** perceived utility $\tilde{u}_i = t \cdot u_i + (1-t) \cdot \eta_i$, where $\eta_i \sim \mathcal{N}(\bar{u}, \sigma_u^2)$.
- **Participation friction $\ell \in [0,1]$:** each voter only votes on their top $K = \max(1, \lceil \ell \cdot m \rceil)$ candidates.

**VSE** (Voter Satisfaction Efficiency) = 0% means as good as random, 100% means always optimal.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from scipy.stats import norm as scipy_norm, binom as scipy_binom
import warnings
warnings.filterwarnings('ignore')

try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False
    print('pandas not found — table output will use plain text')

plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': '#f9f9f9',
                     'axes.grid': True, 'grid.alpha': 0.3})

## Model Settings
Edit this cell before running the simulation.

In [ ]:
# ── Simulation parameters ────────────────────────────────────────────
NV   = 85     # voters
NC   = 8      # candidates
NTR  = 2000    # trials per grid point
NG   = 8      # grid resolution (NG×NG over t and ℓ)
ND   = 2      # spatial dimensions
NORM = 'l1'   # 'l1' (Manhattan), 'l2' (Euclidean), 'linf' (Chebyshev)
SEED = 42     # random seed (None = random each run)

# ── Runoff options ────────────────────────────────────────────
USE_TRUE_RUNOFF = True  # use true utilities for Top 2 finalist runoff

# ── Methods to include ──────────────────────────────────────────
METHODS_ENABLED = {
    'Plurality':        True,
    'Plurality Top 2':  True,
    'Approval':         True,
    'Approval Top 2':   True,
    'RCV':              True,
    'STAR':             True,
    'Condorcet':        True,
    'Score':            False,
    'Borda':            False,
}

# ── Scenarios (t, ℓ) shown in bar charts and hypothesis tests ───────────
SCENARIOS = [
    {'label': 'Ideal (t=1, ℓ=1)',           't': 1.0, 'l': 1.00},
    {'label': 'Low energy (t=1, ℓ=0.35)',   't': 1.0, 'l': 0.35},
    {'label': 'Low knowledge (t=0.3, ℓ=1)', 't': 0.3, 'l': 1.00},
    {'label': 'Both low (t=0.3, ℓ=0.35)',   't': 0.3, 'l': 0.35},
]

ALPHA = 0.01  # hypothesis test significance level

# ── Display order and colors ───────────────────────────────────────
METHOD_ORDER = [
    'Plurality', 'Plurality Top 2', 'Approval', 'Approval Top 2',
    'RCV', 'STAR', 'Condorcet', 'Score', 'Borda',
]
METHOD_COLORS = {
    'Plurality':        '#e05555',
    'Plurality Top 2':  '#cc6a5a',
    'Approval':         '#4ec96a',
    'Approval Top 2':   '#2fa07c',
    'RCV':              '#e09944',
    'STAR':             '#4ab8e0',
    'Condorcet':        '#9b6be0',
    'Score':            '#b8e04a',
    'Borda':            '#d4c94a',
}

## Core Functions

In [ ]:
RNG = np.random.default_rng(SEED)


def make_election(nv, nc, nd=2, norm='l1'):
    v = RNG.uniform(-1, 1, (nv, nd))
    c = RNG.uniform(-1, 1, (nc, nd))
    diff = v[:, np.newaxis, :] - c[np.newaxis, :, :]
    key = str(norm).lower()
    if key == 'l1':
        dist = np.sum(np.abs(diff), axis=2)
    elif key == 'linf':
        dist = np.max(np.abs(diff), axis=2)
    else:
        dist = np.sqrt(np.sum(diff ** 2, axis=2))
    return -(dist ** 2)


def add_noise(u, t):
    if t >= 1.0:
        return u.copy()
    mu, sg = u.mean(), u.std()
    if sg < 1e-9:
        sg = 1.0
    eta = RNG.normal(mu, sg, u.shape)
    return t * u + (1 - t) * eta


def K_of(l, nc):
    return max(1, int(np.ceil(l * nc)))


def top_idx(pu, l):
    K = K_of(l, pu.shape[1])
    return np.argsort(-pu, axis=1)[:, :K]


def vse(sw_w, sw_rand, sw_opt):
    d = sw_opt - sw_rand
    return (sw_w - sw_rand) / d if d > 1e-9 else 1.0

## Voting Methods — Honest

All methods use perceived utilities `pu` and the energy parameter `l`. The `u` argument (true utilities) is only used for Top 2 runoff comparisons when `USE_TRUE_RUNOFF=True`.

In [ ]:
def plurality_winner(pu, **kw):
    top1 = np.argmax(pu, axis=1)
    return int(np.argmax(np.bincount(top1, minlength=pu.shape[1])))


def runoff_plurality_winner(pu, u=None, use_true_runoff=True, **kw):
    top1 = np.argmax(pu, axis=1)
    counts = np.bincount(top1, minlength=pu.shape[1])
    order = np.argsort(-counts)
    f1, f2 = int(order[0]), int(order[1])
    mat = u if (use_true_runoff and u is not None) else pu
    return f1 if np.sum(mat[:, f1] > mat[:, f2]) >= np.sum(mat[:, f2] > mat[:, f1]) else f2


def _approval_ballots(pu, l):
    nv, nc = pu.shape
    tidx = top_idx(pu, l)
    considered = np.zeros((nv, nc), dtype=bool)
    for i in range(nv):
        considered[i, tidx[i]] = True
    gm = pu.mean(axis=1, keepdims=True)
    approvals = considered & (pu > gm)
    for i in np.where(~approvals.any(axis=1))[0]:
        approvals[i, tidx[i, 0]] = True
    return approvals


def approval_winner(pu, l, **kw):
    return int(np.argmax(_approval_ballots(pu, l).sum(axis=0)))


def runoff_approval_winner(pu, l, u=None, use_true_runoff=True, **kw):
    totals = _approval_ballots(pu, l).sum(axis=0)
    order = np.argsort(-totals)
    f1, f2 = int(order[0]), int(order[1])
    mat = u if (use_true_runoff and u is not None) else pu
    return f1 if np.sum(mat[:, f1] > mat[:, f2]) >= np.sum(mat[:, f2] > mat[:, f1]) else f2


def _score_ballots(pu, l):
    nv, nc = pu.shape
    tidx = top_idx(pu, l)
    ballots = np.zeros((nv, nc))
    for i in range(nv):
        row = pu[i]
        lo, hi = row.min(), row.max()
        flat = (hi - lo) < 1e-9
        denom = 1.0 if flat else (hi - lo)
        for c in tidx[i]:
            ballots[i, c] = 5.0 if flat else 5.0 * (row[c] - lo) / denom
    return ballots


def score_winner(pu, l, **kw):
    return int(np.argmax(_score_ballots(pu, l).sum(axis=0)))


def star_winner(pu, l, **kw):
    ballots = _score_ballots(pu, l)
    totals = ballots.sum(axis=0)
    order = np.argsort(-totals)
    f1, f2 = int(order[0]), int(order[1])
    return f1 if np.sum(ballots[:, f1] > ballots[:, f2]) >= np.sum(ballots[:, f2] > ballots[:, f1]) else f2


def irv_winner(pu, l, **kw):
    ballots = top_idx(pu, l)
    nv, K = ballots.shape
    nc = pu.shape[1]
    remaining = list(range(nc))
    remaining_set = set(remaining)
    for _ in range(nc - 1):
        if len(remaining) == 1:
            break
        fp = {c: 0 for c in remaining}
        for i in range(nv):
            for k in range(K):
                c = int(ballots[i, k])
                if c in remaining_set:
                    fp[c] += 1
                    break
        total = sum(fp.values())
        for c, v in fp.items():
            if v > total / 2:
                return c
        elim = min(remaining, key=lambda c: fp.get(c, 0))
        remaining.remove(elim)
        remaining_set.discard(elim)
    return remaining[0]


def borda_winner(pu, l, **kw):
    nv, nc = pu.shape
    tidx = top_idx(pu, l)
    K = tidx.shape[1]
    pts = np.arange(K, 0, -1, dtype=float)
    scores = np.zeros(nc)
    for i in range(nv):
        scores[tidx[i]] += pts
    return int(np.argmax(scores))


def condorcet_winner(pu, l, **kw):
    nv, nc = pu.shape
    tidx = top_idx(pu, l)
    K = tidx.shape[1]
    rm = np.full((nv, nc), K)
    for i in range(nv):
        for k, c in enumerate(tidx[i]):
            rm[i, c] = k
    pw = np.zeros((nc, nc))
    for a in range(nc):
        for b in range(a + 1, nc):
            wa = int(np.sum(rm[:, a] < rm[:, b]))
            wb = int(np.sum(rm[:, b] < rm[:, a]))
            pw[a, b] = wa
            pw[b, a] = wb
    worst = np.zeros(nc)
    for a in range(nc):
        d = pw[:, a] - pw[a, :]
        d[a] = 0
        worst[a] = max(0.0, float(d.max()))
    return int(np.argmin(worst))


METHOD_FNS = {
    'Plurality':        plurality_winner,
    'Plurality Top 2':  runoff_plurality_winner,
    'Approval':         approval_winner,
    'Approval Top 2':   runoff_approval_winner,
    'RCV':              irv_winner,
    'STAR':             star_winner,
    'Condorcet':        condorcet_winner,
    'Score':            score_winner,
    'Borda':            borda_winner,
}
print('Honest methods ready.')

## Voting Methods — Strategic

All strategic voters use polling-based tactical behaviour (strategy share = 100%). Strategies mirror `sim.js`:
- **Plurality/Borda/RCV/Condorcet:** rank or vote to hurt the frontrunner and boost the preferred of {frontrunner, challenger}.
- **Approval:** pivot strategy — approve candidates above the midpoint of frontrunner/challenger utilities.
- **Score/STAR:** rescale scores so preferred of {frontrunner, challenger} gets 5 and the other gets 0.

Both honest and strategic grids use the same RNG seed so they run on identical elections.

In [ ]:
def _front_target(scores):
    order = np.argsort(-np.asarray(scores, dtype=float))
    return int(order[0]), (int(order[1]) if len(order) > 1 else int(order[0]))


# ── Plurality ────────────────────────────────────────────────────────────────────
def strat_plurality_winner(pu, **kw):
    top1 = np.argmax(pu, axis=1)
    counts = np.bincount(top1, minlength=pu.shape[1]).astype(float)
    front, target = _front_target(counts)
    votes = np.where(pu[:, target] > pu[:, front], target, front)
    return int(np.argmax(np.bincount(votes, minlength=pu.shape[1])))


def strat_runoff_plurality_winner(pu, u=None, use_true_runoff=True, **kw):
    top1 = np.argmax(pu, axis=1)
    counts = np.bincount(top1, minlength=pu.shape[1]).astype(float)
    front, target = _front_target(counts)
    votes = np.where(pu[:, target] > pu[:, front], target, front)
    counts2 = np.bincount(votes, minlength=pu.shape[1])
    order = np.argsort(-counts2)
    f1, f2 = int(order[0]), int(order[1])
    mat = u if (use_true_runoff and u is not None) else pu
    return f1 if np.sum(mat[:, f1] > mat[:, f2]) >= np.sum(mat[:, f2] > mat[:, f1]) else f2


# ── Approval (pivot strategy) ────────────────────────────────────────────────
def _strat_approval_ballots(pu, l):
    nv, nc = pu.shape
    tidx = top_idx(pu, l)
    considered = np.zeros((nv, nc), dtype=bool)
    for i in range(nv):
        considered[i, tidx[i]] = True
    # honest poll to find frontrunner/challenger
    gm = pu.mean(axis=1, keepdims=True)
    honest = considered & (pu > gm)
    for i in np.where(~honest.any(axis=1))[0]:
        honest[i, tidx[i, 0]] = True
    front, target = _front_target(honest.sum(axis=0).astype(float))
    # strategic ballots
    approvals = np.zeros((nv, nc), dtype=bool)
    for i in range(nv):
        pivot = (pu[i, front] + pu[i, target]) / 2.0
        row = considered[i] & (pu[i] >= pivot)
        if pu[i, target] > pu[i, front]:
            row[front] = False
            row[target] = considered[i, target]
        else:
            row[target] = False
            row[front] = considered[i, front]
        if not row.any():
            row[tidx[i, 0]] = True
        approvals[i] = row
    return approvals


def strat_approval_winner(pu, l, **kw):
    return int(np.argmax(_strat_approval_ballots(pu, l).sum(axis=0)))


def strat_runoff_approval_winner(pu, l, u=None, use_true_runoff=True, **kw):
    totals = _strat_approval_ballots(pu, l).sum(axis=0)
    order = np.argsort(-totals)
    f1, f2 = int(order[0]), int(order[1])
    mat = u if (use_true_runoff and u is not None) else pu
    return f1 if np.sum(mat[:, f1] > mat[:, f2]) >= np.sum(mat[:, f2] > mat[:, f1]) else f2


# ── Score / STAR (strategic rescaling) ────────────────────────────────────────
def _strat_score_ballots(pu, l):
    nv, nc = pu.shape
    tidx = top_idx(pu, l)
    considered = np.zeros((nv, nc), dtype=bool)
    for i in range(nv):
        considered[i, tidx[i]] = True
    front, target = _front_target(_score_ballots(pu, l).sum(axis=0).astype(float))
    ballots = np.zeros((nv, nc))
    for i in range(nv):
        pref = target if pu[i, target] > pu[i, front] else front
        other = front if pref == target else target
        hi, lo = float(pu[i, pref]), float(pu[i, other])
        if abs(hi - lo) < 1e-12:
            for c in range(nc):
                if considered[i, c] and pu[i, c] >= hi:
                    ballots[i, c] = 5.0
        else:
            for c in range(nc):
                if considered[i, c]:
                    ballots[i, c] = float(np.clip(int((5 + 0.99) * (pu[i, c] - lo) / (hi - lo)), 0, 5))
        if not ballots[i].any():
            ballots[i, tidx[i, 0]] = 5.0
    return ballots


def strat_score_winner(pu, l, **kw):
    return int(np.argmax(_strat_score_ballots(pu, l).sum(axis=0)))


def strat_star_winner(pu, l, **kw):
    ballots = _strat_score_ballots(pu, l)
    order = np.argsort(-ballots.sum(axis=0))
    f1, f2 = int(order[0]), int(order[1])
    return f1 if np.sum(ballots[:, f1] > ballots[:, f2]) >= np.sum(ballots[:, f2] > ballots[:, f1]) else f2


# ── RCV ──────────────────────────────────────────────────────────────────────────────
def strat_irv_winner(pu, l, **kw):
    nv, nc = pu.shape
    tidx = top_idx(pu, l)
    K = tidx.shape[1]
    fp = np.bincount(tidx[:, 0], minlength=nc).astype(float)
    front, target = _front_target(fp)
    poll_desc = list(np.argsort(-fp))
    strat_ballots = []
    for i in range(nv):
        order = list(map(int, tidx[i]))
        order_set = set(order)
        wq = float(pu[i, front])
        by_ltw = [c for c in reversed(poll_desc) if c in order_set and c != front]
        strat = []
        if target in order_set and pu[i, target] > wq:
            strat.append(target)
            by_ltw = [c for c in by_ltw if c != target]
        for c in by_ltw:
            if pu[i, c] > wq:
                strat.append(c)
        if front in order_set:
            strat.append(front)
        for c in by_ltw:
            if pu[i, c] <= wq:
                strat.append(c)
        missing = sorted([c for c in order if c not in set(strat)], key=lambda c: -pu[i, c])
        strat.extend(missing)
        strat_ballots.append(strat[:K])
    remaining = list(range(nc))
    remaining_set = set(remaining)
    for _ in range(nc - 1):
        if len(remaining) == 1:
            break
        votes = {c: 0 for c in remaining}
        for ballot in strat_ballots:
            for c in ballot:
                if c in remaining_set:
                    votes[c] += 1
                    break
        total = sum(votes.values())
        for c, v in votes.items():
            if v > total / 2:
                return c
        elim = min(remaining, key=lambda c: votes.get(c, 0))
        remaining.remove(elim)
        remaining_set.discard(elim)
    return remaining[0]


# ── Borda ────────────────────────────────────────────────────────────────────────────
def strat_borda_winner(pu, l, **kw):
    nv, nc = pu.shape
    tidx = top_idx(pu, l)
    K = tidx.shape[1]
    pts = np.arange(K, 0, -1, dtype=float)
    honest_scores = np.zeros(nc)
    for i in range(nv):
        honest_scores[tidx[i]] += pts
    front, target = _front_target(honest_scores)
    poll_desc = list(np.argsort(-honest_scores))
    scores = np.zeros(nc)
    for i in range(nv):
        order = list(map(int, tidx[i]))
        order_set = set(order)
        if K > 1 and front in order_set and target in order_set:
            middle = list(reversed([c for c in poll_desc if c in order_set and c != front and c != target]))
            order = [target] + middle + [front] if pu[i, target] > pu[i, front] else [front] + middle + [target]
        for k, c in enumerate(order[:K]):
            scores[c] += pts[k]
    return int(np.argmax(scores))


# ── Condorcet (Minimax) ──────────────────────────────────────────────────────────
def strat_condorcet_winner(pu, l, **kw):
    nv, nc = pu.shape
    tidx = top_idx(pu, l)
    K = tidx.shape[1]
    rm = np.full((nv, nc), K)
    for i in range(nv):
        for k, c in enumerate(tidx[i]):
            rm[i, c] = k
    pw = np.zeros((nc, nc))
    for a in range(nc):
        for b in range(a + 1, nc):
            wa = int(np.sum(rm[:, a] < rm[:, b]))
            wb = int(np.sum(rm[:, b] < rm[:, a]))
            pw[a, b] = wa
            pw[b, a] = wb
    polls = np.array([(pw[a] > pw[:, a]).sum() for a in range(nc)], dtype=float)
    front, target = _front_target(polls)
    poll_order = list(np.argsort(-polls))
    for i in range(nv):
        order = list(map(int, tidx[i]))
        order_set = set(order)
        if pu[i, target] > pu[i, front] and front in order_set and target in order_set:
            others = [c for c in poll_order if c in order_set and c != front and c != target]
            ntb = min(float(pu[i, front]), float(pu[i, target]))
            decent = sorted([c for c in others if pu[i, c] >= ntb], key=lambda c: -pu[i, c])
            bad    = sorted([c for c in others if pu[i, c] <  ntb], key=lambda c: -pu[i, c])
            new_order = decent + [target] + bad + [front]
        else:
            others = sorted([c for c in poll_order if c in order_set and c != front], key=lambda c: -pu[i, c])
            new_order = ([front] if front in order_set else []) + others
        rm[i].fill(K)
        for k, c in enumerate(new_order):
            rm[i, c] = k
    pw_s = np.zeros((nc, nc))
    for a in range(nc):
        for b in range(a + 1, nc):
            wa = int(np.sum(rm[:, a] < rm[:, b]))
            wb = int(np.sum(rm[:, b] < rm[:, a]))
            pw_s[a, b] = wa
            pw_s[b, a] = wb
    worst = np.zeros(nc)
    for a in range(nc):
        d = pw_s[:, a] - pw_s[a, :]
        d[a] = 0
        worst[a] = max(0.0, float(d.max()))
    return int(np.argmin(worst))


STRATEGIC_METHOD_FNS = {
    'Plurality':        strat_plurality_winner,
    'Plurality Top 2':  strat_runoff_plurality_winner,
    'Approval':         strat_approval_winner,
    'Approval Top 2':   strat_runoff_approval_winner,
    'RCV':              strat_irv_winner,
    'STAR':             strat_star_winner,
    'Condorcet':        strat_condorcet_winner,
    'Score':            strat_score_winner,
    'Borda':            strat_borda_winner,
}
print('Strategic methods ready.')

## Hypothesis Test Helpers

In [ ]:
def one_sided_mean_test(deltas):
    deltas = np.asarray(deltas, dtype=float)
    n = len(deltas)
    if n < 2:
        return dict(mean=np.nan, sd=np.nan, se=np.nan, p=np.nan, lower_ci=np.nan, upper_ci=np.nan)
    mean = float(np.mean(deltas))
    sd   = float(np.std(deltas, ddof=1))
    se   = sd / np.sqrt(n)
    if sd < 1e-12:
        return dict(mean=mean, sd=sd, se=se, p=0.0 if mean > 0 else 1.0,
                    lower_ci=mean, upper_ci=mean)
    z   = mean / se
    p   = float(np.clip(1 - scipy_norm.cdf(z), 0, 1))
    z99 = float(scipy_norm.ppf(0.995))
    return dict(mean=mean, sd=sd, se=se, p=p,
                lower_ci=mean - z99 * se, upper_ci=mean + z99 * se)


def one_sided_sign_test(deltas):
    deltas = np.asarray(deltas, dtype=float)
    pos    = int(np.sum(deltas >  1e-12))
    neg    = int(np.sum(deltas < -1e-12))
    n_eff  = pos + neg
    if n_eff == 0:
        return dict(pos=0, neg=0, n_eff=0, p=1.0)
    p = float(np.clip(scipy_binom.sf(pos - 1, n_eff, 0.5), 0, 1))
    return dict(pos=pos, neg=neg, n_eff=n_eff, p=p)


print('Hypothesis test helpers ready.')

## Grid Runner & Plot Utilities

In [ ]:
def run_grid(
    nv=NV, nc=NC, ntr=NTR, ng=NG, nd=ND, norm=NORM,
    use_true_runoff=USE_TRUE_RUNOFF,
    methods_enabled=METHODS_ENABLED,
    seed=SEED,
    method_fns=None,
    label='',
):
    global RNG
    RNG = np.random.default_rng(seed)  # reset so honest+strategic use same elections
    if method_fns is None:
        method_fns = METHOD_FNS
    t_vals = np.linspace(0, 1, ng)
    l_vals = np.linspace(1 / nc, 1, ng)
    method_names = [m for m in METHOD_ORDER if methods_enabled.get(m, False)]
    res       = {m: np.zeros((ng, ng))      for m in method_names}
    trial_vse = {m: np.zeros((ng, ng, ntr)) for m in method_names}
    done, total = 0, ng * ng
    tag = f' [{label}]' if label else ''
    print(f'Grid {ng}×{ng}, {ntr} trials, {len(method_names)} methods{tag}')
    for li, l in enumerate(l_vals):
        for ti, t in enumerate(t_vals):
            for tr in range(ntr):
                u  = make_election(nv, nc, nd, norm)
                pu = add_noise(u, t)
                cm = u.mean(axis=0)
                sw_rand, sw_opt = float(cm.mean()), float(cm.max())
                for m in method_names:
                    w = method_fns[m](pu, l=l, u=u, use_true_runoff=use_true_runoff)
                    trial_vse[m][li, ti, tr] = vse(float(cm[w]), sw_rand, sw_opt)
            for m in method_names:
                res[m][li, ti] = trial_vse[m][li, ti, :].mean()
            done += 1
            if done % max(1, total // 8) == 0 or done == total:
                print(f'  {done}/{total} cells', end='\r')
    print(f'\nDone. {ng*ng*ntr:,} total trials.')
    return {'res': res, 'trial_vse': trial_vse, 't_vals': t_vals, 'l_vals': l_vals,
            'method_names': method_names, 'label': label}


def resolve_scenarios(results):
    t_vals, l_vals = results['t_vals'], results['l_vals']
    specs = []
    for sc in SCENARIOS:
        ti = int(np.argmin(np.abs(t_vals - sc['t'])))
        li = int(np.argmin(np.abs(l_vals - sc['l'])))
        specs.append({**sc, 'ti': ti, 'li': li,
                      'actual_t': float(t_vals[ti]), 'actual_l': float(l_vals[li])})
    return specs


# ── Reusable plot functions ──────────────────────────────────────────────────────────
def plot_axis_slices(results):
    res, t_vals, l_vals = results['res'], results['t_vals'], results['l_vals']
    method_names = results['method_names']
    mode_label   = results.get('label', '')
    ng = len(t_vals)
    li_max, ti_max = ng - 1, ng - 1
    fig, axes = plt.subplots(2, 1, figsize=(10, 9))
    # ─ VSE vs t (l = 1)
    ax = axes[0]
    for m in method_names:
        ax.plot(t_vals, res[m][li_max, :], label=m, color=METHOD_COLORS.get(m, '#888'), linewidth=2)
    ax.axhline(1.0, color='gray', linestyle='--', linewidth=0.8, alpha=0.7)
    ax.set_xlabel('Knowledge  t', fontsize=11)
    ax.set_ylabel('VSE', fontsize=11)
    ax.set_title('Full ballot (ℓ = 1) — VSE vs knowledge t', fontsize=11)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax.set_ylim(0, 1.08)
    ax.legend(fontsize=9, loc='lower right')
    # ─ VSE vs l (t = 1)
    ax = axes[1]
    for m in method_names:
        ax.plot(l_vals, res[m][:, ti_max], label=m, color=METHOD_COLORS.get(m, '#888'), linewidth=2)
    ax.axhline(1.0, color='gray', linestyle='--', linewidth=0.8, alpha=0.7)
    ax.set_xlabel('Energy  ℓ', fontsize=11)
    ax.set_ylabel('VSE', fontsize=11)
    ax.set_title('Perfect knowledge (t = 1) — VSE vs energy ℓ', fontsize=11)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax.set_ylim(0, 1.08)
    ax.legend(fontsize=9, loc='lower right')
    plt.suptitle(f'Axis Slices — {mode_label}', fontsize=12)
    plt.tight_layout()
    plt.show()


def plot_scenarios(results, scenario_specs):
    res          = results['res']
    method_names = results['method_names']
    mode_label   = results.get('label', '')
    n_sc = len(scenario_specs)
    fig, axes = plt.subplots(n_sc, 1, figsize=(10, 5 * n_sc))
    if n_sc == 1:
        axes = [axes]
    for ax, sc in zip(axes, scenario_specs):
        vals   = [res[m][sc['li'], sc['ti']] for m in method_names]
        colors = [METHOD_COLORS.get(m, '#888') for m in method_names]
        bars = ax.bar(range(len(method_names)), vals, color=colors, edgecolor='white', linewidth=0.5)
        ax.set_xticks(range(len(method_names)))
        ax.set_xticklabels(method_names, rotation=35, ha='right', fontsize=9)
        ax.set_ylabel('VSE', fontsize=9)
        ax.set_title(f"{sc['label']}  (grid: t={sc['actual_t']:.2f}, ℓ={sc['actual_l']:.2f})", fontsize=10)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
        ax.set_ylim(0, 1.18)
        ax.axhline(1.0, color='gray', linestyle='--', linewidth=0.8, alpha=0.7)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                    f'{v:.1%}', ha='center', va='bottom', fontsize=8)
    plt.suptitle(f'VSE at Scenarios — {mode_label}', fontsize=12)
    plt.tight_layout()
    plt.show()


def plot_scenario_hyp_tests(results, scenario_specs):
    trial_vse    = results['trial_vse']
    method_names = results['method_names']
    mode_label   = results.get('label', '')
    n_sc = len(scenario_specs)
    n_m  = len(method_names)
    z99  = float(scipy_norm.ppf(0.995))
    yes_color, no_color = '#4ec96a', '#e05555'
    cmap = mcolors.ListedColormap([no_color, yes_color])
    cs   = max(0.75, 5.5 / n_m)
    fig, axes = plt.subplots(n_sc, 1, figsize=(n_m * cs + 2.5, n_sc * (n_m * cs + 1.5)))
    if n_sc == 1:
        axes = [axes]
    for ax, sc in zip(axes, scenario_specs):
        li, ti = sc['li'], sc['ti']
        mat   = np.full((n_m, n_m), np.nan)
        means = np.full((n_m, n_m), np.nan)
        for ri, rm in enumerate(method_names):
            for ci, cm_m in enumerate(method_names):
                if ri == ci:
                    continue
                d  = trial_vse[rm][li, ti, :] - trial_vse[cm_m][li, ti, :]
                mn = float(np.mean(d))
                se = float(np.std(d, ddof=1)) / np.sqrt(len(d))
                mat[ri, ci]   = 1.0 if mn - z99 * se > 0 else 0.0
                means[ri, ci] = mn
        ax.imshow(mat, cmap=cmap, vmin=0, vmax=1, aspect='auto')
        ax.set_xticks(range(n_m))
        ax.set_yticks(range(n_m))
        ax.set_xticklabels(method_names, rotation=35, ha='right', fontsize=8)
        ax.set_yticklabels(method_names, fontsize=8)
        ax.set_title(f"{sc['label']}  (t={sc['actual_t']:.2f}, ℓ={sc['actual_l']:.2f})", fontsize=9)
        ax.set_xlabel('Column method (worse?)', fontsize=8)
        ax.set_ylabel('Row method (better?)', fontsize=8)
        for ri in range(n_m):
            for ci in range(n_m):
                if ri == ci:
                    ax.text(ci, ri, '—', ha='center', va='center', fontsize=8, color='#bbb')
                else:
                    txt   = 'Yes' if mat[ri, ci] > 0.5 else 'No'
                    delta = f'{means[ri, ci]:+.1%}' if np.isfinite(means[ri, ci]) else ''
                    ax.text(ci, ri - 0.18, txt,   ha='center', va='center', fontsize=7,   color='white', fontweight='bold')
                    ax.text(ci, ri + 0.28, delta, ha='center', va='center', fontsize=5.5, color='white', alpha=0.9)
    yes_p = mpatches.Patch(color=yes_color, label='Yes — 99% CI for row−col excludes 0')
    no_p  = mpatches.Patch(color=no_color,  label='No  — CI contains 0')
    fig.legend(handles=[yes_p, no_p], loc='lower center', ncol=2, fontsize=9,
               bbox_to_anchor=(0.5, -0.01), framealpha=0.9)
    plt.suptitle(f'Scenario Pairwise Tests — {mode_label}', fontsize=12)
    plt.tight_layout()
    plt.show()


def show_scenario_detail_table(results, scenario_specs):
    trial_vse    = results['trial_vse']
    method_names = results['method_names']
    mode_label   = results.get('label', '')
    rows = []
    for sc in scenario_specs:
        li, ti = sc['li'], sc['ti']
        for rm in method_names:
            for cm_m in method_names:
                if rm == cm_m:
                    continue
                d   = trial_vse[rm][li, ti, :] - trial_vse[cm_m][li, ti, :]
                ms  = one_sided_mean_test(d)
                lo, hi = ms.get('lower_ci', np.nan), ms.get('upper_ci', np.nan)
                rows.append({'Scenario': sc['label'], 'Hypothesis': f'{rm} > {cm_m}',
                             'Mean diff': ms['mean'], 'p': ms['p'],
                             'CI low (99%)': lo, 'CI high (99%)': hi,
                             'CI > 0?': 'Yes' if np.isfinite(lo) and lo > 0 else 'No'})
    print(f'\nScenario detail table — {mode_label}\n')
    if HAS_PANDAS:
        df = pd.DataFrame(rows)
        def fp(x): return f'{x:+.2%}' if np.isfinite(x) else 'N/A'
        def fq(x): return f'{x:.4f}' if np.isfinite(x) else 'N/A'
        df['Mean diff'] = df['Mean diff'].map(fp)
        df['p']         = df['p'].map(fq)
        df['CI low (99%)']  = df['CI low (99%)'].map(fp)
        df['CI high (99%)'] = df['CI high (99%)'].map(fp)
        def hl(row): return ['background-color: #d4edda'] * len(row) if row['CI > 0?'] == 'Yes' else [''] * len(row)
        try:
            display(df.style.apply(hl, axis=1))
        except Exception:
            print(df.to_string(index=False))
    else:
        hdr = f"{'Scenario':<30} {'Hypothesis':<35} {'Mean diff':>10} {'p':>8} {'99% CI':>24} {'CI>0?':>6}"
        print(hdr)
        print('-' * len(hdr))
        for r in rows:
            lo  = f"{r['CI low (99%)']:+.2%}"  if np.isfinite(r['CI low (99%)'])  else 'N/A'
            hi  = f"{r['CI high (99%)']:+.2%}" if np.isfinite(r['CI high (99%)']) else 'N/A'
            md  = f"{r['Mean diff']:+.2%}" if np.isfinite(r['Mean diff']) else 'N/A'
            p   = f"{r['p']:.4f}"         if np.isfinite(r['p'])         else 'N/A'
            print(f"{r['Scenario']:<30} {r['Hypothesis']:<35} {md:>10} {p:>8} [{lo}, {hi}] {r['CI > 0?']:>6}")


print('Grid runner and plot utilities ready.')

## Run Simulation

Both grids reset the RNG to the same seed, so honest and strategic results run on **identical elections** (apples-to-apples comparison).

With default settings (NG=8, NTR=200) each grid takes ~1–3 min.

In [ ]:
results_honest = run_grid(method_fns=METHOD_FNS, label='Honest voting')

In [ ]:
results_strategic = run_grid(method_fns=STRATEGIC_METHOD_FNS, label='Strategic voting')

In [ ]:
# Resolve scenario grid indices (same for both since they share grid parameters)
scenario_specs = resolve_scenarios(results_honest)
for sc in scenario_specs:
    print(f"{sc['label']:35s}  t={sc['actual_t']:.2f}  ℓ={sc['actual_l']:.2f}")

---
## Honest Voting

In [ ]:
plot_axis_slices(results_honest)

In [ ]:
plot_scenarios(results_honest, scenario_specs)

In [ ]:
plot_scenario_hyp_tests(results_honest, scenario_specs)

In [ ]:
show_scenario_detail_table(results_honest, scenario_specs)

---
## Strategic Voting

All voters vote strategically based on a polling-based model (see method notes above). Results on the same elections as honest voting.

In [ ]:
plot_axis_slices(results_strategic)

In [ ]:
plot_scenarios(results_strategic, scenario_specs)

In [ ]:
plot_scenario_hyp_tests(results_strategic, scenario_specs)

In [ ]:
show_scenario_detail_table(results_strategic, scenario_specs)